# AIDS: intersectional bias analysis

**Goal:** Evaluating dataset viability to explore intersectional bias, finding elligible crossing between sensitive attributes and detecting baseline inbalances.

## Setup

AIDS dataset contains the following sensitive attributes: *homo*, *gender*, *race*, *age*. We need to evaluate how combinations of these experiments generate interesting results for intersectional bias. To view insights on the dataset variables: https://archive.ics.uci.edu/dataset/890/aids+clinical+trials+group+study+175

Some binary definitions in the dataset:

```
race: (0=White, 1=non-white)
gender: (0=F, 1=M)
homo: (0=no, 1=yes)
cid: censoring indicator (1 = failure, 0 = censoring)
```

In [ ]:
import sys 
import os

sys.path.append(os.path.abspath("../utils"))
from config import DATA_PATH

# Target variable = censoring indicator (cid). 1 means death, 0 means censored
# already renamed cid to target in pre-processed table
TARGET_COL = "target"
FAVORABLE_LABEL = 0

# Sensitive attributes to evaluate
SENSITIVE_ATTRS = ["race", "gender", "homo"]

# Privileged subgroups
PRIVILEGED =  {
    "gender": 1, # male
    "race": 0, # white
    "homo": 0 # no
}

MIN_SUBGROUP_COUNT = 80

### Visual configurations

In [ ]:
import warnings
warnings. filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors impoort LinearSegmentedColormap

import seaborn as sab

from scipy import stats
from scipy.stats import chi2_contingency

LAB_PALETTE_SEQUENTIAL = [
    "#f0f9fa", # Ice white (Almost white, close to no correlation)
    "#b2ebf2", # Acqua
    "#3eccc4", # Teal (signs start to show up)
    "#1a659e", # Light Blue (moderate association)
    "#002a61"  # Navy Blue (critical association)
]

# global seaborn configurations

sab.set_theme(style="whitegrid")
sab.set_palette(LAB_PALETTE_SEQUENTIAL)

# custom heatmap based on palette
LAB_CMAP = LinearSegmentedColormap.from_list("LabGradient", LAB_PALETTE_SEQUENTIAL)

# LAB_PALETTE_20 for high density (20 colors)
LAB_PALETTE_CATEGORICAL = [
    # NAVY (Higher privilege/status)
    "#001b3d", # Navy Ultra Dark
    "#002a61", # Original Navy
    "#004b8d", # Medium Navy
    "#006cc2", # Shiny Navy
    
    # BLUE (Upper intermediate groups)
    "#134e7a", # Deep Blue
    "#1a659e", # Original Blue
    "#4d8fc1", # Steel Blue
    "#82b4dc", # Sky Blue
    
    # PURPLE (Transition/ Intersection zones)
    "#2e2c8f", # Deep Purple
    "#403eb1", # Purple Original
    "#6b69d6", # Medium Purple
    "#8e8cf2", # Lavender 
    
    # TEAL (Parity groups)
    "#168275", # Deep Teal
    "#1eaf9d", # Teal Original
    "#28d1bd", # Bright Teal
    "#5fe3d3", # Mint Teal
    
    # AQUA (Higher vulnerability)
    "#2fa39d", # Dark Aqua
    "#3eccc4", # Aqua Original
    "#66cbe1", # Sky Blue Original
    "#b3b1ff"  # Pale Lavender (lightest point)
]

sns.set_palette(LAB_PALETTE_CATEGORICAL)

plt.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
})